# Wyckoff LLaMA+LoRA Backend on Google Colab

Use this notebook as the temporary cloud GPU backend for the Vercel app.

Before running: Runtime -> Change runtime type -> T4 GPU.

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## Clone project and install dependencies

In [ ]:
%cd /content
!rm -rf Wyckoff_Trading_App
!git clone https://github.com/JiachengCui-01/Wyckoff_Trading_App.git
%cd /content/Wyckoff_Trading_App
!python -m pip install -q -r requirements-ai.txt

## Hugging Face token and LoRA adapter

Recommended: put your `HF_TOKEN` in Colab Secrets, then upload/copy your LoRA adapter folder to Google Drive at:

`/content/drive/MyDrive/wyckoff_models/llama_wyckoff_lora`

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

os.environ['LLAMA_BASE_MODEL'] = os.environ.get('LLAMA_BASE_MODEL', 'meta-llama/Llama-2-7b-hf')
os.environ['LLAMA_LORA_PATH'] = '/content/drive/MyDrive/wyckoff_models/llama_wyckoff_lora'

print('LLAMA_BASE_MODEL =', os.environ['LLAMA_BASE_MODEL'])
print('LLAMA_LORA_PATH =', os.environ['LLAMA_LORA_PATH'])
print('LoRA exists:', os.path.exists(os.environ['LLAMA_LORA_PATH']))

## Build offline RAG index

This downloads the embedding model once and writes `data/rag_index/*`.

In [ ]:
%cd /content/Wyckoff_Trading_App
!python scripts/build_rag_index.py

## Start FastAPI backend

If your LoRA adapter is missing, set `WYCKOFF_LLM_MOCK=1` for a connectivity smoke test. For real LLaMA+LoRA inference, keep it `0`.

In [ ]:
import os, subprocess, time, pathlib, textwrap

lora_path = pathlib.Path(os.environ['LLAMA_LORA_PATH'])
os.environ['WYCKOFF_LLM_MOCK'] = '0' if lora_path.exists() else '1'
print('WYCKOFF_LLM_MOCK =', os.environ['WYCKOFF_LLM_MOCK'])

server = subprocess.Popen([
    'python', '-m', 'uvicorn', 'ai_backend.main:app',
    '--host', '0.0.0.0', '--port', '8000'
], cwd='/content/Wyckoff_Trading_App', env=os.environ.copy())
time.sleep(8)
!curl -s http://127.0.0.1:8000/health

## Expose backend with Cloudflare Quick Tunnel

Copy the printed `https://...trycloudflare.com` URL. That is your temporary `LLAMA_BACKEND_URL` for Vercel. It changes whenever this Colab runtime restarts.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared tunnel --url http://127.0.0.1:8000

## Test through the public tunnel

Replace `YOUR_TUNNEL_URL` with the `https://...trycloudflare.com` URL from the previous cell.

In [ ]:
TUNNEL_URL = 'YOUR_TUNNEL_URL'
if TUNNEL_URL.startswith('https://'):
    !curl -s $TUNNEL_URL/health
    !curl -s -X POST $TUNNEL_URL/chat -H 'Content-Type: application/json' -d '{"question":"What is a Spring in Wyckoff methodology?"}'
else:
    print('Paste your tunnel URL first.')

## Connect Vercel

In Vercel project settings, set:

`LLAMA_BACKEND_URL = https://...trycloudflare.com`

Then redeploy Vercel. The app's lower-left status should change to `LLaMA+LoRA Backend Online`.